In [ ]:
# pip install "numpy<2"

  Using cached numpy-1.26.4-cp310-cp310-win_amd64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp310-cp310-win_amd64.whl (15.8 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6
Note: you may need to restart the kernel to use updated packages.


  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mediapipe 0.10.21 requires protobuf<5,>=4.25.3, but you have protobuf 3.19.6 which is incompatible.
onnx 1.19.1 requires protobuf>=4.25.1, but you have protobuf 3.19.6 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
streamlit 1.51.0 requires protobuf<7,>=3.20, but you have protobuf 3.19.6 which is incompatible.
tf-keras 2.20.1 requires tensorflow<2.21,>=2.20, but you have tensorflow 2.10.1 which is incompatible.


In [ ]:
# pip install "tensorflow<2.11"


   ---------------------------------------- 0.0/455.9 MB ? eta -:--:--
   ---------------------------------------- 1.0/455.9 MB 7.2 MB/s eta 0:01:04
   ---------------------------------------- 2.9/455.9 MB 8.0 MB/s eta 0:00:57
   ---------------------------------------- 5.2/455.9 MB 9.4 MB/s eta 0:00:49
    --------------------------------------- 8.4/455.9 MB 10.8 MB/s eta 0:00:42
    --------------------------------------- 10.7/455.9 MB 11.2 MB/s eta 0:00:40
   - -------------------------------------- 13.4/455.9 MB 11.5 MB/s eta 0:00:39
   - -------------------------------------- 16.3/455.9 MB 11.8 MB/s eta 0:00:38
   - -------------------------------------- 18.4/455.9 MB 11.9 MB/s eta 0:00:37
   - -------------------------------------- 20.4/455.9 MB 11.3 MB/s eta 0:00:39
   - -------------------------------------- 21.8/455.9 MB 10.9 MB/s eta 0:00:40
   - -------------------------------------- 22.5/455.9 MB 10.3 MB/s eta 0:00:42
   -- ------------------------------------- 23.6/455.9 

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mediapipe 0.10.21 requires numpy<2, but you have numpy 2.2.6 which is incompatible.
mediapipe 0.10.21 requires protobuf<5,>=4.25.3, but you have protobuf 3.19.6 which is incompatible.
onnx 1.19.1 requires protobuf>=4.25.1, but you have protobuf 3.19.6 which is incompatible.
streamlit 1.51.0 requires protobuf<7,>=3.20, but you have protobuf 3.19.6 which is incompatible.
tf-keras 2.20.1 requires tensorflow<2.21,>=2.20, but you have tensorflow 2.10.1 which is incompatible.


## HuggingFace Gender Classification

Link: https://huggingface.co/rizvandwiki/gender-classification

In [17]:
import os
import torch
import pandas as pd
import numpy as np
from PIL import Image
from tqdm import tqdm


# ============================================================================
# MODEL 1: HuggingFace ViT Gender Classifier (GPU + Batch Optimized)
# ============================================================================
class HuggingFaceGenderClassifier:
    def __init__(self, model_name="rizvandwiki/gender-classification"):
        from transformers import AutoImageProcessor, AutoModelForImageClassification
        
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        print(f"Loading {model_name} on {self.device}...")
        self.processor = AutoImageProcessor.from_pretrained(model_name)
        self.model = AutoModelForImageClassification.from_pretrained(model_name)
        self.model = self.model.to(self.device)
        self.model.eval()
        print("✓ Loaded successfully.")

    def predict_batch(self, image_paths):
        """
        image_paths: list of paths
        Returns: list of (gender, confidence)
        """

        # Load images
        images = []
        for p in image_paths:
            try:
                img = Image.open(p).convert("RGB")
                images.append(img)
            except:
                images.append(None)

        # Remove None values (missing images)
        valid_indices = [i for i, img in enumerate(images) if img is not None]
        valid_images = [images[i] for i in valid_indices]

        if len(valid_images) == 0:
            return [(None, None) for _ in image_paths]

        # Preprocess batch
        inputs = self.processor(images=valid_images, return_tensors="pt")
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        # Forward pass
        with torch.no_grad():
            outputs = self.model(**inputs)
            probs = torch.nn.functional.softmax(outputs.logits, dim=-1).cpu().numpy()

        # Map predictions back
        results = [(None, None)] * len(image_paths)
        for j, i in enumerate(valid_indices):
            p = probs[j]
            cls = p.argmax()
            conf = float(p[cls])
            gender_raw = self.model.config.id2label[cls]
            gender = gender_raw.capitalize().replace("_", " ").replace("portrait", "").strip()
            results[i] = (gender, conf)

        return results


# ============================================================================
# EVALUATION FUNCTION (SIMPLE, FULLY SELF-CONTAINED)
# ============================================================================
def evaluate_huggingface(csv_path, image_root, output_csv, batch_size=32):
    df = pd.read_csv(csv_path, dtype={"subject_id": str})
    df["subject_id"] = df["subject_id"].str.zfill(4)

    df = df[df["gender"].notna()]                # removes NaN
    df = df[df["gender"].astype(str) != ""]      # removes empty string
    df = df[df["gender"].astype(str) != "nan"]   # extra safety

    model = HuggingFaceGenderClassifier()

    all_true = []
    all_pred = []
    all_conf = []
    all_tone = []

    # Collect image paths
    image_paths = []
    true_labels = []
    tones = []

    for _, row in df.iterrows():
        img = os.path.join(image_root, row["subject_id"], row["cropped_image"])
        if os.path.exists(img):
            image_paths.append(img)
            true_labels.append(row["gender"].lower().strip())
            tones.append(int(row["mst_label"]))

    # Batch inference
    print("\nRunning batched inference...")
    predictions = []

    for i in tqdm(range(0, len(image_paths), batch_size)):
        batch_paths = image_paths[i:i+batch_size]
        batch_preds = model.predict_batch(batch_paths)
        predictions.extend(batch_preds)

    # Build results
    for (pred_gender, conf), true_g, t in zip(predictions, true_labels, tones):
        if pred_gender is None:
            continue
        all_true.append(true_g)
        all_pred.append(pred_gender.lower())
        all_conf.append(conf)
        all_tone.append(t)

    # Save detailed results
    result_df = pd.DataFrame({
        "image_path": image_paths[:len(all_true)],
        "true_gender": all_true,
        "pred_gender": all_pred,
        "confidence": all_conf,
        "mst_label": all_tone
    })
    result_df.to_csv(output_csv, index=False)
    print(f"\nSaved predictions → {output_csv}")

    # Metrics
    def acc(y, yp): return np.mean(np.array(y) == np.array(yp))
    def acc_class(y, yp, c):
        idx = [i for i, v in enumerate(y) if v == c]
        if len(idx) == 0: return 0
        return np.mean([yp[j] == c for j in idx])

    def confmat(y, yp):
        TP = sum((t=="male") and (p=="male") for t,p in zip(y,yp))
        TN = sum((t=="female") and (p=="female") for t,p in zip(y,yp))
        FP = sum((t=="female") and (p=="male") for t,p in zip(y,yp))
        FN = sum((t=="male") and (p=="female") for t,p in zip(y,yp))
        return TP, TN, FP, FN

    metrics = {
        "overall_accuracy": acc(all_true, all_pred),
        "male_accuracy": acc_class(all_true, all_pred, "male"),
        "female_accuracy": acc_class(all_true, all_pred, "female")
    }

    TP, TN, FP, FN = confmat(all_true, all_pred)
    metrics.update({"TP":TP, "TN":TN, "FP":FP, "FN":FN})

    # Per-tone
    per_tone = {}
    unique_tones = sorted(set(all_tone))
    for tone in unique_tones:
        idx = [i for i,t in enumerate(all_tone) if t==tone]
        yt = [all_true[j] for j in idx]
        yp = [all_pred[j] for j in idx]

        per_tone[tone] = {
            "N": len(idx),
            "accuracy": acc(yt, yp),
            "male_accuracy": acc_class(yt, yp, "male"),
            "female_accuracy": acc_class(yt, yp, "female")
        }

    return metrics, per_tone, result_df


In [18]:
csv_path = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\ccv2_per_image_filenames_only.csv"
image_root = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2"
output_csv = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\hf_gender_eval.csv"

hf_metrics, hf_per_tone, hf_result_df = evaluate_huggingface(csv_path, image_root, output_csv, batch_size=32)

Loading rizvandwiki/gender-classification on cuda...
✓ Loaded successfully.

Running batched inference...


100%|██████████| 5557/5557 [26:45<00:00,  3.46it/s] 



Saved predictions → G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\hf_gender_eval.csv


In [19]:
hf_metrics

{'overall_accuracy': 0.7153487064116986,
 'male_accuracy': 0.9626209322779243,
 'female_accuracy': 0.5614071910932652,
 'TP': 65670,
 'TN': 61519,
 'FP': 48061,
 'FN': 2550}

In [20]:
hf_per_tone

{1: {'N': 1190,
  'accuracy': 0.6579831932773109,
  'male_accuracy': 0.9901315789473685,
  'female_accuracy': 0.5440180586907449},
 2: {'N': 13337,
  'accuracy': 0.7027067556421984,
  'male_accuracy': 0.9744773282650013,
  'female_accuracy': 0.5990263103376838},
 3: {'N': 33730,
  'accuracy': 0.703172250222354,
  'male_accuracy': 0.9755395683453237,
  'female_accuracy': 0.5813344775799185},
 4: {'N': 38214,
  'accuracy': 0.6833359501753284,
  'male_accuracy': 0.9690738083851427,
  'female_accuracy': 0.5436730559451457},
 5: {'N': 55796,
  'accuracy': 0.7603233206681482,
  'male_accuracy': 0.9629658875552748,
  'female_accuracy': 0.5918668767231193},
 6: {'N': 23073,
  'accuracy': 0.7139080310319421,
  'male_accuracy': 0.937828883258624,
  'female_accuracy': 0.534540629146827},
 7: {'N': 6438,
  'accuracy': 0.6609195402298851,
  'male_accuracy': 0.9445993031358885,
  'female_accuracy': 0.4327354260089686},
 8: {'N': 4183,
  'accuracy': 0.7121683002629692,
  'male_accuracy': 0.9762695775

## Realistic Gender Classifier

Link: https://huggingface.co/prithivMLmods/Realistic-Gender-Classification

In [25]:
import os
import torch
import pandas as pd
import numpy as np
from PIL import Image
from tqdm import tqdm


# ============================================================================
# MODEL 2: Realistic Gender Classification (SigLIP-based, GPU + Batching)
# ============================================================================
class RealisticGenderClassifier:
    """
    prithivMLmods/Realistic-Gender-Classification
    SigLIP backbone — very strong performance
    """

    def __init__(self):
        from transformers import AutoImageProcessor, AutoModelForImageClassification
        
        model_name = "prithivMLmods/Realistic-Gender-Classification"
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        print(f"Loading {model_name} on {self.device}...")
        self.processor = AutoImageProcessor.from_pretrained(model_name)
        self.model = AutoModelForImageClassification.from_pretrained(model_name)
        self.model = self.model.to(self.device)
        self.model.eval()
        print("✓ Model loaded successfully.")

    def predict_batch(self, image_paths):
        """
        image_paths: list[str]
        Returns: list of (gender, confidence)
        """

        # Load images
        images = []
        for p in image_paths:
            try:
                img = Image.open(p).convert("RGB")
                images.append(img)
            except:
                images.append(None)

        valid_idx = [i for i, img in enumerate(images) if img is not None]
        valid_images = [images[i] for i in valid_idx]

        if len(valid_images) == 0:
            return [(None, None) for _ in image_paths]

        # Preprocess
        inputs = self.processor(images=valid_images, return_tensors="pt")
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        # Forward pass
        with torch.no_grad():
            outputs = self.model(**inputs)
            probs = torch.nn.functional.softmax(outputs.logits, dim=-1).cpu().numpy()

        # Map predictions
        results = [(None, None)] * len(image_paths)
        for j, i in enumerate(valid_idx):
            p = probs[j]
            cls = p.argmax()
            conf = float(p[cls])

            # Label format: "male portrait" → take first token
            label = self.model.config.id2label[cls]
            gender = label.split()[0].capitalize()

            results[i] = (gender, conf)

        return results


# ============================================================================
# EVALUATION FUNCTION (SELF-CONTAINED)
# ============================================================================
def evaluate_realistic(csv_path, image_root, output_csv, batch_size=32):
    df = pd.read_csv(csv_path, dtype={"subject_id": str})
    df["subject_id"] = df["subject_id"].str.zfill(4)

    df = df[df["gender"].notna()]                # removes NaN
    df = df[df["gender"].astype(str) != ""]      # removes empty string
    df = df[df["gender"].astype(str) != "nan"]   # extra safety

    model = RealisticGenderClassifier()

    all_true = []
    all_pred = []
    all_conf = []
    all_tone = []

    # Collect paths
    image_paths = []
    true_labels = []
    tones = []

    for _, row in df.iterrows():
        img = os.path.join(image_root, row["subject_id"], row["cropped_image"])
        if os.path.exists(img):
            image_paths.append(img)
            true_labels.append(row["gender"].lower())
            tones.append(int(row["mst_label"]))

    # Batch inference
    print("\nRunning batched inference (Realistic Gender Model)...")
    predictions = []

    for i in tqdm(range(0, len(image_paths), batch_size)):
        batch_paths = image_paths[i:i+batch_size]
        batch_preds = model.predict_batch(batch_paths)
        predictions.extend(batch_preds)

    # Build results
    for (pred_g, conf), tg, t in zip(predictions, true_labels, tones):
        if pred_g is None:
            continue
        all_true.append(tg)
        all_pred.append(pred_g.lower())
        all_conf.append(conf)
        all_tone.append(t)

    # Save results CSV
    out_df = pd.DataFrame({
        "image_path": image_paths[:len(all_true)],
        "true_gender": all_true,
        "pred_gender": all_pred,
        "confidence": all_conf,
        "mst_label": all_tone
    })
    out_df.to_csv(output_csv, index=False)
    print(f"\nSaved Realistic Model predictions → {output_csv}")

    # Metrics
    def acc(y, yp): return np.mean(np.array(y) == np.array(yp))
    def acc_class(y, yp, c):
        idx = [i for i, v in enumerate(y) if v == c]
        if len(idx) == 0: return 0
        return np.mean([yp[j] == c for j in idx])

    def confmat(y, yp):
        TP = sum((t=="male")   and (p=="male")   for t,p in zip(y,yp))
        TN = sum((t=="female") and (p=="female") for t,p in zip(y,yp))
        FP = sum((t=="female") and (p=="male")   for t,p in zip(y,yp))
        FN = sum((t=="male")   and (p=="female") for t,p in zip(y,yp))
        return TP, TN, FP, FN

    metrics = {
        "overall_accuracy": acc(all_true, all_pred),
        "male_accuracy": acc_class(all_true, all_pred, "male"),
        "female_accuracy": acc_class(all_true, all_pred, "female")
    }

    TP, TN, FP, FN = confmat(all_true, all_pred)
    metrics.update({"TP":TP, "TN":TN, "FP":FP, "FN":FN})

    # Per-tone
    per_tone = {}
    for tone in sorted(set(all_tone)):
        idx = [i for i,t in enumerate(all_tone) if t==tone]
        yt = [all_true[j] for j in idx]
        yp = [all_pred[j] for j in idx]

        per_tone[tone] = {
            "N": len(idx),
            "accuracy": acc(yt, yp),
            "male_accuracy": acc_class(yt, yp, "male"),
            "female_accuracy": acc_class(yt, yp, "female")
        }

    return metrics, per_tone, out_df


In [26]:
csv_path = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\ccv2_per_image_filenames_only.csv"
image_root = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2"
output_csv = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\realistic_gender_eval.csv"

realistic_metrics, realistic_per_tone, realistic_result_df = evaluate_realistic(csv_path, image_root, output_csv, batch_size=32)

print(realistic_metrics)
print(realistic_per_tone)
print(realistic_result_df.head())

Loading prithivMLmods/Realistic-Gender-Classification on cuda...
✓ Model loaded successfully.

Running batched inference (Realistic Gender Model)...


100%|██████████| 5559/5559 [22:30<00:00,  4.12it/s]



Saved Realistic Model predictions → G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\realistic_gender_eval.csv
{'overall_accuracy': 0.885810989356977, 'male_accuracy': 0.9242473102518249, 'female_accuracy': 0.8618947291615363, 'TP': 63054, 'TN': 94499, 'FP': 15142, 'FN': 5168}
{1: {'N': 1190, 'accuracy': 0.7563025210084033, 'male_accuracy': 0.9769736842105263, 'female_accuracy': 0.6805869074492099}, 2: {'N': 13337, 'accuracy': 0.8702106920596836, 'male_accuracy': 0.9679609014390442, 'female_accuracy': 0.8329189973068158}, 3: {'N': 33732, 'accuracy': 0.8933060595280445, 'male_accuracy': 0.9458085555342414, 'female_accuracy': 0.869818930747447}, 4: {'N': 38273, 'accuracy': 0.8932406657434745, 'male_accuracy': 0.9394277516537818, 'female_accuracy': 0.8707144523050611}, 5: {'N': 55797, 'accuracy': 0.9056400881767837, 'male_accuracy': 0.9182722678458622, 'female_accuracy': 0.8951393219337688}, 6: {'N': 23074, 'accuracy': 0.8793880558204039, 'male_accuracy': 0.8843305398557786, 'female

## DeepFace

Link: https://github.com/serengil/deepface

In [27]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm


# ============================================================================
# DEEPFACE MODEL (NO BATCHING — DEEPFACE DOES NOT SUPPORT BATCH INPUT)
# ============================================================================
class DeepFaceClassifier:
    """
    DeepFace gender classifier
    Requires:
        pip install deepface tensorflow
    Runs CPU-only unless your TensorFlow is GPU-enabled.
    """

    def __init__(self):
        try:
            from deepface import DeepFace
            self.deepface = DeepFace
            print("✓ DeepFace loaded successfully.")
        except ImportError:
            print("✗ DeepFace not installed. Run: pip install deepface tensorflow")
            self.deepface = None

    def predict_single(self, image_path):
        if self.deepface is None:
            return None, None

        try:
            result = self.deepface.analyze(
                img_path=image_path,
                actions=['gender'],
                detector_backend='opencv',
                enforce_detection=False,
                silent=True
            )

            # DeepFace sometimes returns list of results
            if isinstance(result, list):
                result = result[0]

            gender = result['dominant_gender'].capitalize()
            confidence = float(result['gender'][result['dominant_gender']])

            return gender, confidence

        except Exception:
            return None, None


# ============================================================================
# EVALUATION FUNCTION (SELF-CONTAINED)
# ============================================================================
def evaluate_deepface(csv_path, image_root, output_csv):
    df = pd.read_csv(csv_path, dtype={"subject_id": str})
    df["subject_id"] = df["subject_id"].str.zfill(4)

    df = df[df["gender"].notna()]                # removes NaN
    df = df[df["gender"].astype(str) != ""]      # removes empty string
    df = df[df["gender"].astype(str) != "nan"]   # extra safety

    model = DeepFaceClassifier()

    all_true = []
    all_pred = []
    all_conf = []
    all_tone = []
    image_paths = []

    print("\nCollecting image paths...")
    for _, row in df.iterrows():
        img = os.path.join(image_root, row["subject_id"], row["cropped_image"])
        if os.path.exists(img):
            image_paths.append(img)
            all_true.append(row["gender"].lower().strip())
            all_tone.append(int(row["mst_label"]))

    print(f"Found {len(image_paths)} images to process.")

    # Run DeepFace sequentially
    print("\nRunning DeepFace inference...")
    predictions = []
    for img in tqdm(image_paths):
        pred_gender, conf = model.predict_single(img)
        predictions.append((pred_gender, conf))

    # Filter out invalid predictions
    final_true = []
    final_pred = []
    final_conf = []
    final_tone = []
    final_paths = []

    for (pg, cf), tg, t, p in zip(predictions, all_true, all_tone, image_paths):
        if pg is None:
            continue

        final_true.append(tg)
        final_pred.append(pg.lower())
        final_conf.append(cf)
        final_tone.append(t)
        final_paths.append(p)

    # Save results to CSV
    result_df = pd.DataFrame({
        "image_path": final_paths,
        "true_gender": final_true,
        "pred_gender": final_pred,
        "confidence": final_conf,
        "mst_label": final_tone
    })
    result_df.to_csv(output_csv, index=False)
    print(f"\nSaved DeepFace predictions → {output_csv}")

    # ----------------------------------------
    # METRICS
    # ----------------------------------------
    def acc(y, yp): return np.mean(np.array(y) == np.array(yp))

    def acc_class(y, yp, c):
        idx = [i for i,v in enumerate(y) if v == c]
        if len(idx) == 0:
            return 0.0
        return np.mean([yp[j] == c for j in idx])

    def confusion(y, yp):
        TP = sum((t=="male")   and (p=="male")   for t,p in zip(y,yp))
        TN = sum((t=="female") and (p=="female") for t,p in zip(y,yp))
        FP = sum((t=="female") and (p=="male")   for t,p in zip(y,yp))
        FN = sum((t=="male")   and (p=="female") for t,p in zip(y,yp))
        return TP, TN, FP, FN

    metrics = {
        "overall_accuracy": acc(final_true, final_pred),
        "male_accuracy": acc_class(final_true, final_pred, "male"),
        "female_accuracy": acc_class(final_true, final_pred, "female"),
    }

    TP, TN, FP, FN = confusion(final_true, final_pred)
    metrics.update({"TP":TP, "TN":TN, "FP":FP, "FN":FN})

    # Per-tone accuracy
    per_tone = {}
    for tone in sorted(set(final_tone)):
        idx = [i for i,t in enumerate(final_tone) if t == tone]
        yt = [final_true[j] for j in idx]
        yp = [final_pred[j] for j in idx]

        per_tone[tone] = {
            "N": len(idx),
            "accuracy": acc(yt, yp),
            "male_accuracy": acc_class(yt, yp, "male"),
            "female_accuracy": acc_class(yt, yp, "female")
        }

    return metrics, per_tone, result_df


In [28]:
csv_path = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\ccv2_per_image_filenames_only.csv"
image_root = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2"
output_csv = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\deepface_gender_eval.csv"

deepface_metrics, deepface_per_tone, deepface_result_df = evaluate_deepface(csv_path, image_root, output_csv)

print(deepface_metrics)
print(deepface_per_tone)
print(deepface_result_df.head())

✓ DeepFace loaded successfully.

Found 177863 images to process.

Running DeepFace inference...


 13%|█▎        | 23005/177863 [1:11:15<7:59:40,  5.38it/s] 


KeyboardInterrupt: 

## InsightFace

Link: https://github.com/deepinsight/insightface

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from insightface.app import FaceAnalysis


# ============================================================================
# INSIGHTFACE MODEL (PADDING + SINGLE IMAGE PREDICTION)
# ============================================================================
class InsightFaceGenderClassifier:

    def __init__(self, det_size=(640, 640), pad_amount=1050):
        self.pad_amount = pad_amount
        try:
            self.app = FaceAnalysis(providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
        except:
            self.app = FaceAnalysis(providers=['CPUExecutionProvider'])
        
        self.app.prepare(ctx_id=0, det_size=det_size)
        print("✓ InsightFace initialized.")

    def predict_single(self, image_path):
        img = cv2.imread(image_path)
        if img is None:
            return None, None

        # Pad image -> This is done as the zoomed-in faces in CCV2 cause detection issues
        pad = self.pad_amount
        img_padded = cv2.copyMakeBorder(
            img,
            pad, pad, pad, pad,
            cv2.BORDER_CONSTANT,
            value=[0, 0, 0]
        )

        faces = self.app.get(img_padded)

        if len(faces) == 0:
            return None, None

        face = faces[0]
        gender = "Male" if face.gender == 1 else "Female"

        # InsightFace does NOT provide a clean confidence score → mock using soft probability
        # But face.genderprobability exists in some builds
        try:
            conf = float(face.gender_probability)
        except:
            conf = 1.0  # assume high confidence for segmented CCV2 images

        return gender, conf


# ============================================================================
# EVALUATION FUNCTION
# ============================================================================
def evaluate_insightface(csv_path, image_root, output_csv):
    df = pd.read_csv(csv_path, dtype={"subject_id": str})
    df["subject_id"] = df["subject_id"].str.zfill(4)

    df = df[df["gender"].notna()]                # removes NaN
    df = df[df["gender"].astype(str) != ""]      # removes empty string
    df = df[df["gender"].astype(str) != "nan"]   # extra safety

    model = InsightFaceGenderClassifier()

    all_true, all_pred, all_conf, all_tone, paths = [], [], [], [], []

    print("\nRunning InsightFace inference...")

    for _, row in tqdm(df.iterrows(), total=len(df)):

        img_path = os.path.join(image_root, row["subject_id"], row["cropped_image"])

        if not os.path.exists(img_path):
            continue

        pred_gender, conf = model.predict_single(img_path)
        if pred_gender is None:
            continue

        all_true.append(row["gender"].lower())
        all_pred.append(pred_gender.lower())
        all_conf.append(conf)
        all_tone.append(int(row["mst_label"]))
        paths.append(img_path)

    # Save results
    out_df = pd.DataFrame({
        "image_path": paths,
        "true_gender": all_true,
        "pred_gender": all_pred,
        "confidence": all_conf,
        "mst_label": all_tone,
    })
    out_df.to_csv(output_csv, index=False)
    print(f"Saved InsightFace predictions → {output_csv}")

    # Metrics
    def acc(y, yp): return np.mean(np.array(y) == np.array(yp))
    def acc_class(y, yp, lbl):
        idx = [i for i, g in enumerate(y) if g == lbl]
        if len(idx) == 0: return 0.0
        return np.mean([yp[j] == lbl for j in idx])

    def confmat(y, yp):
        TP = sum((t=="male")   and (p=="male")   for t,p in zip(y,yp))
        TN = sum((t=="female") and (p=="female") for t,p in zip(y,yp))
        FP = sum((t=="female") and (p=="male")   for t,p in zip(y,yp))
        FN = sum((t=="male")   and (p=="female") for t,p in zip(y,yp))
        return TP, TN, FP, FN

    metrics = {
        "overall_accuracy": acc(all_true, all_pred),
        "male_accuracy": acc_class(all_true, all_pred, "male"),
        "female_accuracy": acc_class(all_true, all_pred, "female")
    }
    TP, TN, FP, FN = confmat(all_true, all_pred)
    metrics.update({"TP":TP, "TN":TN, "FP":FP, "FN":FN})

    # Per-tone
    per_tone = {}
    for tone in sorted(set(all_tone)):
        idx = [i for i, t in enumerate(all_tone) if t == tone]
        yt = [all_true[j] for j in idx]
        yp = [all_pred[j] for j in idx]
        per_tone[tone] = {
            "N": len(idx),
            "accuracy": acc(yt, yp),
            "male_accuracy": acc_class(yt, yp, "male"),
            "female_accuracy": acc_class(yt, yp, "female")
        }

    return metrics, per_tone, out_df


c:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\DatasetAnnotation\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from insightface.app import FaceAnalysis


# ============================================================================
# INSIGHTFACE MODEL (PADDING + BATCH PREDICTION)
# ============================================================================
class InsightFaceGenderClassifier:

    def __init__(self, det_size=(640, 640), pad_amount=1050, batch_size=32):
        self.pad_amount = pad_amount
        self.batch_size = batch_size
        try:
            self.app = FaceAnalysis(providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
        except:
            self.app = FaceAnalysis(providers=['CPUExecutionProvider'])
        
        self.app.prepare(ctx_id=0, det_size=det_size)
        print(f"✓ InsightFace initialized with batch_size={batch_size}.")

    def predict_batch(self, image_paths):
        """Process multiple images at once."""
        results = []
        
        for img_path in image_paths:
            img = cv2.imread(img_path)
            if img is None:
                results.append((None, None))
                continue

            # Pad image
            pad = self.pad_amount
            img_padded = cv2.copyMakeBorder(
                img,
                pad, pad, pad, pad,
                cv2.BORDER_CONSTANT,
                value=[0, 0, 0]
            )

            faces = self.app.get(img_padded)

            if len(faces) == 0:
                results.append((None, None))
                continue

            face = faces[0]
            gender = "Male" if face.gender == 1 else "Female"

            try:
                conf = float(face.gender_probability)
            except:
                conf = 1.0

            results.append((gender, conf))
        
        return results


# ============================================================================
# EVALUATION FUNCTION WITH BATCHING
# ============================================================================
def evaluate_insightface(csv_path, image_root, output_csv, batch_size=32):
    df = pd.read_csv(csv_path, dtype={"subject_id": str})
    df["subject_id"] = df["subject_id"].str.zfill(4)

    df = df[df["gender"].notna()]                # removes NaN
    df = df[df["gender"].astype(str) != ""]      # removes empty string
    df = df[df["gender"].astype(str) != "nan"]   # extra safety

    model = InsightFaceGenderClassifier(batch_size=batch_size)

    all_true, all_pred, all_conf, all_tone, paths = [], [], [], [], []

    print(f"\nRunning InsightFace inference with batch_size={batch_size}...")

    # Prepare batches
    batch_paths = []
    batch_rows = []
    
    for _, row in df.iterrows():
        img_path = os.path.join(image_root, row["subject_id"], row["cropped_image"])
        if os.path.exists(img_path):
            batch_paths.append(img_path)
            batch_rows.append(row)

    # Process in batches
    num_batches = (len(batch_paths) + batch_size - 1) // batch_size
    
    for i in tqdm(range(0, len(batch_paths), batch_size), total=num_batches):
        batch_end = min(i + batch_size, len(batch_paths))
        current_paths = batch_paths[i:batch_end]
        current_rows = batch_rows[i:batch_end]
        
        predictions = model.predict_batch(current_paths)
        
        for path, row, (pred_gender, conf) in zip(current_paths, current_rows, predictions):
            if pred_gender is None:
                continue
                
            all_true.append(row["gender"].lower())
            all_pred.append(pred_gender.lower())
            all_conf.append(conf)
            all_tone.append(int(row["mst_label"]))
            paths.append(path)

    # Save results
    out_df = pd.DataFrame({
        "image_path": paths,
        "true_gender": all_true,
        "pred_gender": all_pred,
        "confidence": all_conf,
        "mst_label": all_tone,
    })
    out_df.to_csv(output_csv, index=False)
    print(f"Saved InsightFace predictions → {output_csv}")

    # Metrics (unchanged)
    def acc(y, yp): return np.mean(np.array(y) == np.array(yp))
    def acc_class(y, yp, lbl):
        idx = [i for i, g in enumerate(y) if g == lbl]
        if len(idx) == 0: return 0.0
        return np.mean([yp[j] == lbl for j in idx])

    def confmat(y, yp):
        TP = sum((t=="male")   and (p=="male")   for t,p in zip(y,yp))
        TN = sum((t=="female") and (p=="female") for t,p in zip(y,yp))
        FP = sum((t=="female") and (p=="male")   for t,p in zip(y,yp))
        FN = sum((t=="male")   and (p=="female") for t,p in zip(y,yp))
        return TP, TN, FP, FN

    metrics = {
        "overall_accuracy": acc(all_true, all_pred),
        "male_accuracy": acc_class(all_true, all_pred, "male"),
        "female_accuracy": acc_class(all_true, all_pred, "female")
    }
    TP, TN, FP, FN = confmat(all_true, all_pred)
    metrics.update({"TP":TP, "TN":TN, "FP":FP, "FN":FN})

    # Per-tone
    per_tone = {}
    for tone in sorted(set(all_tone)):
        idx = [i for i, t in enumerate(all_tone) if t == tone]
        yt = [all_true[j] for j in idx]
        yp = [all_pred[j] for j in idx]
        per_tone[tone] = {
            "N": len(idx),
            "accuracy": acc(yt, yp),
            "male_accuracy": acc_class(yt, yp, "male"),
            "female_accuracy": acc_class(yt, yp, "female")
        }

    return metrics, per_tone, out_df

In [8]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from insightface.app import FaceAnalysis
from concurrent.futures import ThreadPoolExecutor, as_completed


# ============================================================================
# INSIGHTFACE MODEL (PADDING + PARALLEL BATCH PREDICTION)
# ============================================================================
class InsightFaceGenderClassifier:

    def __init__(self, det_size=(640, 640), pad_amount=1050, batch_size=32, num_workers=4):
        self.pad_amount = pad_amount
        self.batch_size = batch_size
        self.num_workers = num_workers
        try:
            self.app = FaceAnalysis(providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
        except:
            self.app = FaceAnalysis(providers=['CPUExecutionProvider'])
        
        self.app.prepare(ctx_id=0, det_size=det_size)
        print(f"✓ InsightFace initialized with batch_size={batch_size}, num_workers={num_workers}.")

    def _process_single_image(self, img_path):
        """Helper method to process a single image."""
        img = cv2.imread(img_path)
        if img is None:
            return None, None

        # Pad image
        pad = self.pad_amount
        img_padded = cv2.copyMakeBorder(
            img,
            pad, pad, pad, pad,
            cv2.BORDER_CONSTANT,
            value=[0, 0, 0]
        )

        faces = self.app.get(img_padded)

        if len(faces) == 0:
            return None, None

        face = faces[0]
        gender = "Male" if face.gender == 1 else "Female"

        try:
            conf = float(face.gender_probability)
        except:
            conf = 1.0

        return gender, conf

    def predict_batch(self, image_paths):
        """Process multiple images in parallel using ThreadPoolExecutor."""
        results = [None] * len(image_paths)
        
        # Use ThreadPoolExecutor for parallel I/O and CPU-bound operations
        with ThreadPoolExecutor(max_workers=self.num_workers) as executor:
            # Submit all tasks
            future_to_idx = {
                executor.submit(self._process_single_image, img_path): idx 
                for idx, img_path in enumerate(image_paths)
            }
            
            # Collect results as they complete
            for future in as_completed(future_to_idx):
                idx = future_to_idx[future]
                try:
                    results[idx] = future.result()
                except Exception as e:
                    results[idx] = (None, None)
        
        return results


# ============================================================================
# EVALUATION FUNCTION WITH PARALLEL BATCHING
# ============================================================================
def evaluate_insightface(csv_path, image_root, output_csv, batch_size=32, num_workers=4):
    """
    Evaluate InsightFace model with parallel batch processing.
    
    Args:
        csv_path: Path to CSV with image metadata
        image_root: Root directory for images
        output_csv: Output path for predictions
        batch_size: Number of images to process per batch
        num_workers: Number of parallel workers (threads) for processing
    """
    df = pd.read_csv(csv_path, dtype={"subject_id": str})
    df["subject_id"] = df["subject_id"].str.zfill(4)

    df = df[df["gender"].notna()]                # removes NaN
    df = df[df["gender"].astype(str) != ""]      # removes empty string
    df = df[df["gender"].astype(str) != "nan"]   # extra safety

    model = InsightFaceGenderClassifier(batch_size=batch_size, num_workers=num_workers)

    all_true, all_pred, all_conf, all_tone, paths = [], [], [], [], []

    print(f"\nRunning InsightFace inference with batch_size={batch_size}, {num_workers} workers...")

    # Prepare batches
    batch_paths = []
    batch_rows = []
    
    for _, row in df.iterrows():
        img_path = os.path.join(image_root, row["subject_id"], row["cropped_image"])
        if os.path.exists(img_path):
            batch_paths.append(img_path)
            batch_rows.append(row)

    # Process in batches with parallel execution
    num_batches = (len(batch_paths) + batch_size - 1) // batch_size
    
    for i in tqdm(range(0, len(batch_paths), batch_size), total=num_batches):
        batch_end = min(i + batch_size, len(batch_paths))
        current_paths = batch_paths[i:batch_end]
        current_rows = batch_rows[i:batch_end]
        
        # Process batch in parallel
        predictions = model.predict_batch(current_paths)
        
        for path, row, (pred_gender, conf) in zip(current_paths, current_rows, predictions):
            if pred_gender is None:
                continue
                
            all_true.append(row["gender"].lower())
            all_pred.append(pred_gender.lower())
            all_conf.append(conf)
            all_tone.append(int(row["mst_label"]))
            paths.append(path)

    # Save results
    out_df = pd.DataFrame({
        "image_path": paths,
        "true_gender": all_true,
        "pred_gender": all_pred,
        "confidence": all_conf,
        "mst_label": all_tone,
    })
    out_df.to_csv(output_csv, index=False)
    print(f"Saved InsightFace predictions → {output_csv}")

    # Metrics (unchanged)
    def acc(y, yp): return np.mean(np.array(y) == np.array(yp))
    def acc_class(y, yp, lbl):
        idx = [i for i, g in enumerate(y) if g == lbl]
        if len(idx) == 0: return 0.0
        return np.mean([yp[j] == lbl for j in idx])

    def confmat(y, yp):
        TP = sum((t=="male")   and (p=="male")   for t,p in zip(y,yp))
        TN = sum((t=="female") and (p=="female") for t,p in zip(y,yp))
        FP = sum((t=="female") and (p=="male")   for t,p in zip(y,yp))
        FN = sum((t=="male")   and (p=="female") for t,p in zip(y,yp))
        return TP, TN, FP, FN

    metrics = {
        "overall_accuracy": acc(all_true, all_pred),
        "male_accuracy": acc_class(all_true, all_pred, "male"),
        "female_accuracy": acc_class(all_true, all_pred, "female")
    }
    TP, TN, FP, FN = confmat(all_true, all_pred)
    metrics.update({"TP":TP, "TN":TN, "FP":FP, "FN":FN})

    # Per-tone
    per_tone = {}
    for tone in sorted(set(all_tone)):
        idx = [i for i, t in enumerate(all_tone) if t == tone]
        yt = [all_true[j] for j in idx]
        yp = [all_pred[j] for j in idx]
        per_tone[tone] = {
            "N": len(idx),
            "accuracy": acc(yt, yp),
            "male_accuracy": acc_class(yt, yp, "male"),
            "female_accuracy": acc_class(yt, yp, "female")
        }

    return metrics, per_tone, out_df

In [9]:
csv_path = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\ccv2_per_image_filenames_only.csv"
image_root = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2"
output_csv = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\insightface_gender_eval.csv"

insightface_metrics, insightface_per_tone, insightface_result_df = evaluate_insightface(csv_path, image_root, output_csv, batch_size=32)

print(insightface_metrics)
print(insightface_per_tone)
print(insightface_result_df.head())

Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CUDAExecutionProvider': {'device_id': '0', 'has_user_compute_stream': '0', 'cudnn_conv1d_pad_to_nc1d': '0', 'user_compute_stream': '0', 'gpu_external_alloc': '0', 'gpu_mem_limit': '18446744073709551615', 'enable_cuda_graph': '0', 'gpu_external_free': '0', 'gpu_external_empty_cache': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'do_copy_in_default_stream': '1', 'cudnn_conv_use_max_workspace': '1', 'tunable_op_enable': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'prefer_nhwc': '0', 'use_ep_level_unified_stream': '0', 'use_tf32': '1', 'sdpa_kernel': '0', 'fuse_conv_bias': '0'}, 'CPUExecutionProvider': {}}
find model: C:\Users\User/.insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']

  0%|          | 0/5559 [00:00<?, ?it/s]c:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\DatasetAnnotation\.venv\lib\site-packages\insightface\utils\transform.py:68: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  P = np.linalg.lstsq(X_homo, Y)[0].T # Affine matrix. 3 x 4
100%|██████████| 5559/5559 [45:20<00:00,  2.04it/s]


Saved InsightFace predictions → G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\insightface_gender_eval.csv
{'overall_accuracy': 0.7708239881817475, 'male_accuracy': 0.7841286762400451, 'female_accuracy': 0.7626171297057373, 'TP': 50018, 'TN': 78863, 'FP': 24548, 'FN': 13770}
{1: {'N': 1135, 'accuracy': 0.7242290748898679, 'male_accuracy': 0.8047945205479452, 'female_accuracy': 0.6963226571767497}, 2: {'N': 12761, 'accuracy': 0.7824621894835828, 'male_accuracy': 0.7859810399310543, 'female_accuracy': 0.7811422413793103}, 3: {'N': 32350, 'accuracy': 0.7962287480680061, 'male_accuracy': 0.8159243993163768, 'female_accuracy': 0.7874838191313663}, 4: {'N': 36456, 'accuracy': 0.7763056835637481, 'male_accuracy': 0.7818406933553446, 'female_accuracy': 0.7735507990633088}, 5: {'N': 52279, 'accuracy': 0.7780753266129804, 'male_accuracy': 0.7979871209859696, 'female_accuracy': 0.7618799861255636}, 6: {'N': 20932, 'accuracy': 0.7376743741639595, 'male_accuracy': 0.7124090761046574, 'female

## FairFace

Link: https://github.com/dchen236/FairFace

In [4]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from transformers import pipeline


# ============================================================================
# FAIRFACE MODEL (HUGGINGFACE PIPELINE WITH BATCHING)
# ============================================================================
class FairFaceGenderClassifier:

    def __init__(self, batch_size=32):
        print("Loading FairFace model...")
        self.batch_size = batch_size
        self.classifier = pipeline(
            "image-classification",
            model="dima806/fairface_gender_image_detection",
            device=0 if torch.cuda.is_available() else -1,
            batch_size=batch_size  # Enable batching in pipeline
        )
        print(f"✓ FairFace loaded with batch_size={batch_size}.")

    def predict_batch(self, image_paths):
        """Process multiple images at once."""
        images = []
        valid_indices = []
        
        for idx, img_path in enumerate(image_paths):
            try:
                img = Image.open(img_path).convert("RGB")
                images.append(img)
                valid_indices.append(idx)
            except:
                pass
        
        if not images:
            return [(None, None)] * len(image_paths)
        
        try:
            # Process all images in batch
            results = self.classifier(images)
            
            # Map results back to original indices
            predictions = [(None, None)] * len(image_paths)
            for valid_idx, result in zip(valid_indices, results):
                label = result[0]["label"]
                conf = float(result[0]["score"])
                predictions[valid_idx] = (label, conf)
            
            return predictions
        except:
            return [(None, None)] * len(image_paths)


# ============================================================================
# EVALUATION FUNCTION WITH BATCHING
# ============================================================================
def evaluate_fairface(csv_path, image_root, output_csv, batch_size=32):
    df = pd.read_csv(csv_path, dtype={"subject_id": str})
    df["subject_id"] = df["subject_id"].str.zfill(4)

    df = df[df["gender"].notna()]                # removes NaN
    df = df[df["gender"].astype(str) != ""]      # removes empty string
    df = df[df["gender"].astype(str) != "nan"]   # extra safety

    model = FairFaceGenderClassifier(batch_size=batch_size)

    all_true, all_pred, all_conf, all_tone, paths = [], [], [], [], []

    print(f"\nRunning FairFace inference with batch_size={batch_size}...")

    # Prepare batches
    batch_paths = []
    batch_rows = []
    
    for _, row in df.iterrows():
        img_path = os.path.join(image_root, row["subject_id"], row["cropped_image"])
        if os.path.exists(img_path):
            batch_paths.append(img_path)
            batch_rows.append(row)

    # Process in batches
    num_batches = (len(batch_paths) + batch_size - 1) // batch_size
    
    for i in tqdm(range(0, len(batch_paths), batch_size), total=num_batches):
        batch_end = min(i + batch_size, len(batch_paths))
        current_paths = batch_paths[i:batch_end]
        current_rows = batch_rows[i:batch_end]
        
        predictions = model.predict_batch(current_paths)
        
        for path, row, (pred_gender, conf) in zip(current_paths, current_rows, predictions):
            if pred_gender is None:
                continue
                
            all_true.append(row["gender"].lower())
            all_pred.append(pred_gender.lower())
            all_conf.append(conf)
            all_tone.append(int(row["mst_label"]))
            paths.append(path)

    # Save CSV
    out_df = pd.DataFrame({
        "image_path": paths,
        "true_gender": all_true,
        "pred_gender": all_pred,
        "confidence": all_conf,
        "mst_label": all_tone,
    })
    out_df.to_csv(output_csv, index=False)
    print(f"Saved FairFace predictions → {output_csv}")

    # Metrics (unchanged)
    def acc(y, yp): return np.mean(np.array(y) == np.array(yp))
    def acc_class(y, yp, lbl):
        idx = [i for i, g in enumerate(y) if g == lbl]
        if len(idx) == 0: return 0
        return np.mean([yp[j] == lbl for j in idx])

    def confmat(y, yp):
        TP = sum((t=="male")   and (p=="male")   for t,p in zip(y,yp))
        TN = sum((t=="female") and (p=="female") for t,p in zip(y,yp))
        FP = sum((t=="female") and (p=="male")   for t,p in zip(y,yp))
        FN = sum((t=="male")   and (p=="female") for t,p in zip(y,yp))
        return TP, TN, FP, FN

    metrics = {
        "overall_accuracy": acc(all_true, all_pred),
        "male_accuracy": acc_class(all_true, all_pred, "male"),
        "female_accuracy": acc_class(all_true, all_pred, "female")
    }
    TP, TN, FP, FN = confmat(all_true, all_pred)
    metrics.update({"TP":TP, "TN":TN, "FP":FP, "FN":FN})

    # Per-tone breakdown
    per_tone = {}
    for tone in sorted(set(all_tone)):
        idx = [i for i, t in enumerate(all_tone) if t == tone]
        yt = [all_true[j] for j in idx]
        yp = [all_pred[j] for j in idx]
        per_tone[tone] = {
            "N": len(idx),
            "accuracy": acc(yt, yp),
            "male_accuracy": acc_class(yt, yp, "male"),
            "female_accuracy": acc_class(yt, yp, "female")
        }

    return metrics, per_tone, out_df

In [5]:
csv_path = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\ccv2_per_image_filenames_only.csv"
image_root = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2"
output_csv = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\fairface_gender_eval.csv"

fairface_metrics, fairface_per_tone, fairface_result_df = evaluate_fairface(csv_path, image_root, output_csv)

print(fairface_metrics)
print(fairface_per_tone)
print(fairface_result_df.head())

Loading FairFace model...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Device set to use cuda:0


✓ FairFace loaded with batch_size=32.

Running FairFace inference with batch_size=32...


100%|██████████| 5559/5559 [24:30<00:00,  3.78it/s]  


Saved FairFace predictions → G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\fairface_gender_eval.csv
{'overall_accuracy': 0.8006330715213394, 'male_accuracy': 0.983216557708657, 'female_accuracy': 0.6870240147390119, 'TP': 67077, 'TN': 75326, 'FP': 34315, 'FN': 1145}
{1: {'N': 1190, 'accuracy': 0.6840336134453782, 'male_accuracy': 1.0, 'female_accuracy': 0.5756207674943566}, 2: {'N': 13337, 'accuracy': 0.7647896828372198, 'male_accuracy': 0.9915829486831388, 'female_accuracy': 0.6782680754091568}, 3: {'N': 33732, 'accuracy': 0.7890726906201826, 'male_accuracy': 0.9904085938998657, 'female_accuracy': 0.6990045481850168}, 4: {'N': 38273, 'accuracy': 0.7833982180649544, 'male_accuracy': 0.9857336415079302, 'female_accuracy': 0.6847158516675736}, 5: {'N': 55797, 'accuracy': 0.8290409878667312, 'male_accuracy': 0.9827858496525584, 'female_accuracy': 0.701237323180938}, 6: {'N': 23074, 'accuracy': 0.8106960214960561, 'male_accuracy': 0.9690118885207561, 'female_accuracy': 0.6838901030

### Result

The best performing model appears to be the **Realistic Gender Classifier** as it's the most accurate and a relatively fast model, producign the following metrics:

- Overall Accuracy: 0.886
    - Male Accuracy: 0.924
    - Female Accuracy': 0.862

- Confusion Matrix:
    - TP: 63054
    - TN: 94499
    - FP: 15142
    - FN: 5168

- Per SkinTone Accuracy:

    - **Skin Tone 1**  
        - N = 1,190  
        - Accuracy: **0.756**  
            - Male Accuracy: 0.977  
            - Female Accuracy: 0.681  

    - **Skin Tone 2**  
        - N = 13,337  
        - Accuracy: **0.870**  
            - Male Accuracy: 0.968  
            - Female Accuracy: 0.833  

    - **Skin Tone 3**  
        - N = 33,732  
        - Accuracy: **0.893**  
            - Male Accuracy: 0.946  
            - Female Accuracy: 0.870  

    - **Skin Tone 4**  
        - N = 38,273  
        - Accuracy: **0.893**  
            - Male Accuracy: 0.939  
            - Female Accuracy: 0.871  

    - **Skin Tone 5**  
        - N = 55,797  
        - Accuracy: **0.906**  
            - Male Accuracy: 0.918  
            - Female Accuracy: 0.895  

    - **Skin Tone 6**  
        - N = 23,074  
        - Accuracy: **0.879**  
            - Male Accuracy: 0.884  
            - Female Accuracy: 0.875  

    - **Skin Tone 7**  
        - N = 6,438  
        - Accuracy: **0.826**  
            - Male Accuracy: 0.906  
            - Female Accuracy: 0.761  

    - **Skin Tone 8**  
        - N = 4,183  
        - Accuracy: **0.803**  
            - Male Accuracy: 0.925  
            - Female Accuracy: 0.680  

    - **Skin Tone 9**  
        - N = 1,671  
        - Accuracy: **0.662**  
            - Male Accuracy: 0.948  
            - Female Accuracy: 0.488  

    - **Skin Tone 10**  
        - N = 168  
        - Accuracy: **0.726**  
            - Male Accuracy: 1.000  
            - Female Accuracy: 0.570  
